
## 1. Prediction Problem and Target Definition

### Intended business objective

The proposed objective is to help AtliQ Credit Intelligence estimate the risk of loan default using information that would be available before a lending decision.

This is a portfolio-project hypothesis, not a validated banking use case.

### Proposed prediction point

- **Prediction point:** before loan approval or disbursement.
- **Candidate target:** `default` from `loans.csv`.
- **Prediction horizon:** unknown.
- **Target-generation logic:** undocumented.
- **Target observation date:** unknown.
- **Dataset documentation:** unavailable.

### Questions that must be resolved

1. What does `default = True` mean in this dataset?
2. Does the target describe a future default or an existing status?
3. Over what period is default observed?
4. When were customer and bureau attributes collected?
5. Does each feature represent information available at the proposed prediction point?

### Current modelling decision

The `default` column is a candidate target only. No model training or final feature selection will occur until the target meaning and feature timing have been investigated.

### Evidence and limitations

The raw audit reports 50,000 loan records:
- `default = False`: 45,703 records.
- `default = True`: 4,297 records.
- Observed positive-label proportion: 8.594%.

These counts describe the supplied labels. They do not establish the target's business definition, observation horizon, or suitability for future-default prediction.



## 2. Candidate Feature Inventory and Leakage Risks

### Proposed prediction point

Before loan approval or disbursement. This is a provisional project assumption.

### Customer attributes

Potential candidates: `age`, `gender`, `marital_status`, `employment_status`, `income`, `number_of_dependants`, `residence_type`, `years_at_current_address`.

`city`, `state`, and `zipcode` require additional usefulness, privacy, and generalization review. `cust_id` is retained for joins and excluded from model inputs.

All candidate attributes require confirmation that they were available at the prediction point.

### Loan attributes

- `loan_purpose`, `loan_type`: potentially eligible if known at application time.
- `sanction_amount`, `loan_amount`, `loan_tenure_months`: verify whether these represent requested or approved terms.
- `bank_balance_at_application`: potentially eligible, subject to verification.
- `processing_fee`, `gst`, `net_disbursement`: high timing risk because they may be determined during or after approval.
- `principal_outstanding`: high leakage risk because it may represent a later account state.
- `disbursal_date`, `installment_start_dt`: event dates requiring timing review.
- `default`: candidate target only, never a feature when predicting default.
- `loan_id`, `cust_id`: identifiers, not predictive features.

### Bureau attributes

`number_of_open_accounts`, `number_of_closed_accounts`, `total_loan_months`, `delinquent_months`, `total_dpd`, `enquiry_count`, and `credit_utilization_ratio` require verification of their snapshot date and measurement window.

`delinquent_months` and `total_dpd` need particular scrutiny for overlap with the target observation period.

### Limitations and next actions

These classifications are preliminary judgments based on column names and the proposed prediction point, not verified data lineage.

Next:
1. Investigate suspicious records and possible formula exceptions.
2. Document unresolved target and feature-timing questions.
3. Define a defensible feature policy before cleaning or model training.
4. Preserve raw files and keep transformations reproducible.



## 3. Suspicious Record Investigation

### 3.1 Net disbursement relationship

**Test performed:** Compared `net_disbursement` with `0.80 * loan_amount` across all 50,000 loan records, allowing an absolute difference of 0.01.

**Observed result:**
- Records tested: 50,000
- Records matching within tolerance: 50,000
- Exceptions: 0

**Interpretation:** The supplied dataset follows an exact 80% relationship within the tested tolerance. This suggests a consistent data-generation rule, but its business rationale is undocumented.

**Decision:** Record the relationship as an observed pattern. Do not infer that the formula is a verified banking rule. Review `loan_amount` and `net_disbursement` for prediction-time eligibility before modeling.

### 3.2 Zero-loan records

**Test performed:** Inspected all records with `loan_amount = 0`.

**Observed result:**
- Records: 8
- `sanction_amount`, `processing_fee`, `gst`, `net_disbursement`, and `bank_balance_at_application` are zero in all eight.
- `principal_outstanding = -1` in all eight.
- `default = False` in all eight.

**Interpretation:** These records share a consistent nonstandard pattern. Their operational meaning is unknown because dataset documentation is unavailable.

**Decision:** Flag these records for investigation. Do not overwrite the raw values or assume that `-1` is definitively a missing-value sentinel. Exclusion from a future modeling dataset requires an explicit, justified rule.

### 3.3 Reproducibility and limitations

The investigation read `data/raw/loans.csv` and printed results without writing to the raw CSV.

The findings describe this supplied dataset only. They do not establish the target definition, feature availability dates, or suitability for model training.



## 4. Outstanding Balance and Processing Fee Investigation

### 4.1 Principal outstanding maximum

**Observed result:**
- Maximum `principal_outstanding`: 5,000,000.
- Records at the maximum: 1,840 (3.68% of 50,000).
- Next-highest distinct balance: 190,799.
- Mean `loan_amount` among records at the maximum: approximately 19.53 million.
- Mean `loan_amount` below the maximum: approximately 3.41 million.
- Both `default = True` and `default = False` occur in the maximum-balance group.

**Interpretation:** The repeated maximum and large gap from the next-highest distinct value warrant investigation. Possible capping or clipping is a hypothesis, not a confirmed explanation.

**Decision:** Preserve the raw values. Flag `principal_outstanding` for both data-quality investigation and prediction-time leakage review. Do not automatically cap, remove, or transform it.

### 4.2 Processing fee anomalies

**Observed result:**
- Seven records have `processing_fee > loan_amount`.
- Their fee-to-loan ratios range from approximately 1.07 to 1.49.
- All seven have `default = False`.
- All seven still satisfy the observed 80% relationship between `net_disbursement` and `loan_amount`.

**Interpretation:** These records are unusual under the apparent interpretation of processing fees, but the fee definition, units, and business rules are undocumented.

**Decision:** Flag the records for investigation. Do not overwrite or remove their values based solely on the ratio. Review the fee fields for prediction-time eligibility before modeling.

### 4.3 Overall decision

The current evidence supports documenting suspicious patterns, not declaring confirmed errors. Raw data remains unchanged. Any future transformation or exclusion must have an explicit rationale and be reproducible.
